# Getting to Know a Dataset — Global Unemployment

A medium-level data exploration case study inspired by DataCamp: validate, summarize, and visualize a country-level economic dataset using **Pandas + Seaborn**.

**Case study:** global unemployment in 2021.

## Learning Map

`Initial exploration → categorical counts → numerical validation → categorical validation → range validation → groupby/agg → named aggregations → Seaborn visualization`

## 1. Initial exploration
Use `head()`, `shape`, `info()`, and `describe()` to understand structure, size, missingness, and numerical distributions before analysing a dataset.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

continents = {
    "Australia": "Oceania", "Japan": "Asia", "South Korea": "Asia",
    "India": "Asia", "China": "Asia", "Germany": "Europe",
    "France": "Europe", "United Kingdom": "Europe", "United States": "North America",
    "Canada": "North America", "Brazil": "South America", "Argentina": "South America",
    "Nigeria": "Africa", "South Africa": "Africa", "Egypt": "Africa"
}
base = {"Asia": 5.8, "Europe": 7.1, "Oceania": 4.8, "North America": 6.2, "South America": 10.4, "Africa": 11.2}
rows=[]
for country, continent in continents.items():
    unemployment=max(1.5, rng.normal(base[continent], 1.8))
    population_m=rng.uniform(5, 1400)
    gdp_pc=rng.lognormal(np.log(25000), .65)
    rows.append({"country":country,"continent":continent,"year":2021,"unemployment_rate":round(unemployment,2),"population_m":round(population_m,1),"gdp_per_capita":round(gdp_pc,0)})
df=pd.DataFrame(rows)
df.head()

Matplotlib is building the font cache; this may take a moment.


,country,continent,year,unemployment_rate,population_m,gdp_per_capita
0,Australia,Oceania,2021,5.35,617.2,40718.0
1,Japan,Asia,2021,7.49,136.4,10724.0
2,South Korea,Asia,2021,6.03,1101.6,24728.0
3,India,Asia,2021,4.26,522.3,41448.0
4,China,Asia,2021,5.92,1152.8,33878.0


## 2. Functions for initial exploration
Create a reusable profiling function so the same exploration workflow can be applied to future datasets.

In [2]:
class DatasetProfiler:
    def __init__(self, data):
        self.data = data

    def overview(self):
        return pd.Series({
            "rows": len(self.data),
            "columns": self.data.shape[1],
            "missing_cells": int(self.data.isna().sum().sum()),
            "duplicate_rows": int(self.data.duplicated().sum()),
        })

    def numeric_summary(self):
        return self.data.describe(numeric_only=True).T

profiler = DatasetProfiler(df)
profiler.overview()

rows              15
columns            6
missing_cells      0
duplicate_rows     0
dtype: int64

## 3. Counting categorical values
`value_counts()` reveals category frequency and is an immediate check for unexpected labels or highly imbalanced categories.

In [3]:
print(df["continent"].value_counts())
print("\nUnique countries:", df["country"].nunique())

continent
Asia             4
Europe           3
Africa           3
North America    2
South America    2
Oceania          1
Name: count, dtype: int64

Unique countries: 15


## 4. Global unemployment in 2021
Filter the target year and calculate an overall unemployment statistic. A weighted average is often more meaningful than a simple country average when populations differ substantially.

In [4]:
global_simple = df["unemployment_rate"].mean()
global_weighted = np.average(df["unemployment_rate"], weights=df["population_m"])
print(f"Simple country average: {global_simple:.2f}%")
print(f"Population-weighted estimate: {global_weighted:.2f}%")

Simple country average: 7.87%
Population-weighted estimate: 8.23%


## 5. Data validation
Validation checks whether the data satisfies assumptions required for analysis. Separate structural checks from domain-specific checks.

In [5]:
assert df["year"].notna().all()
assert df["country"].notna().all()
assert df["unemployment_rate"].notna().all()
print("Basic required-field validation passed.")

Basic required-field validation passed.
